In [0]:
```python
# Databricks Auto Loader - Schema Evolution

# ============================================================
# Step 01 - Setup
# ============================================================

catalog = "retail"
schema = "sales"
volume = "autoloader_demo"

base = f"/Volumes/{catalog}/{schema}/{volume}/schema_evolution"

landing = f"{base}/landing"
schema_loc = f"{base}/_schema"
checkpoint = f"{base}/_checkpoint"

table_name = f"{catalog}.{schema}.orders_schema_evolution"

dbutils.fs.mkdirs(landing)

print("Schema evolution environment created.")


# ============================================================
# Step 02 - Create initial JSON file
# ============================================================

dbutils.fs.put(
    f"{landing}/orders_01.json",
    '{"order_id": 1, "customer": "Aditi", "amount": 250}\n'
    '{"order_id": 2, "customer": "Rahul", "amount": 120}\n',
    True
)

print("Initial JSON file created.")


# ============================================================
# Step 03 - Define Auto Loader
# ============================================================

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing)
)

print("Auto Loader stream defined.")


# ============================================================
# Step 04 - Initial ingestion
# ============================================================

query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(table_name)
)

print("Initial data loaded.")


# ============================================================
# Step 05 - Check initial data
# ============================================================

display(
    spark.sql(f"""
        SELECT *
        FROM {table_name}
        ORDER BY order_id
    """)
)


# ============================================================
# Step 06 - Add a new column
# ============================================================

dbutils.fs.put(
    f"{landing}/orders_02.json",
    '{"order_id": 3, "customer": "Meera", "amount": 890, "channel": "Online"}\n',
    True
)

print("New file with channel column added.")


# ============================================================
# Step 07 - Process the changed schema
# ============================================================

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing)
)

query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(table_name)
)

print("Schema evolution processed.")


# ============================================================
# Step 08 - Check evolved schema
# ============================================================

display(
    spark.sql(f"""
        SELECT *
        FROM {table_name}
        ORDER BY order_id
    """)
)


# ============================================================
# Step 09 - Add unexpected data type
# ============================================================

dbutils.fs.put(
    f"{landing}/orders_03.json",
    '{"order_id": 4, "customer": "Sanjay", "amount": "NOT_A_NUMBER", "channel":_
```
